# Toxic Comment Detection (Jupyter Notebook Edition)

This notebook reproduces the full toxic comment detection workflow—data loading, preprocessing, feature extraction, model training, hyper-parameter tuning, evaluation, and inference—in a single, self-contained environment. Run the notebook top-to-bottom inside Jupyter or JupyterLab to generate all artefacts and visualisations.


## 1. Environment Setup

The following cell imports every library used throughout the workflow, defines project-wide constants, and ensures that output folders exist for metrics and figures.


In [ ]:
import os
from pathlib import Path
import re
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

RANDOM_STATE = 42
DATA_PATH = Path("train.csv")
ARTIFACTS_DIR = Path("artifacts")
RESULTS_DIR = Path("results")

ARTIFACTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

np.random.seed(RANDOM_STATE)


## 2. Load the Dataset

The Kaggle-style dataset ships with the repository as `train.csv`. It contains two columns: `comment_text` and the binary target `toxic`. We normalise the column names for convenience and perform a quick integrity check.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [c.strip().lower() for c in df_raw.columns]
if not {"comment_text", "toxic"}.issubset(df_raw.columns):
    raise ValueError("Expected 'comment_text' and 'toxic' columns in train.csv")

df_raw = df_raw[["comment_text", "toxic"]].copy()
df_raw["toxic"] = df_raw["toxic"].astype(int)
print(df_raw.shape)
df_raw.head()


## 3. Text Preprocessing Utilities

We replicate the five-stage cleaning pipeline from the original project: lower-casing, punctuation removal, tokenisation, stop-word removal, and lemmatisation. Each helper function is pure so that the transformations can be chained or reused individually.


In [ ]:
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()


def lowercase(text: str) -> str:
    return str(text).lower()


def remove_punctuation(text: str) -> str:
    return re.sub(r"[^a-z\s]", "", str(text))


def tokenize(text: str) -> List[str]:
    return str(text).split()


def remove_stopwords(tokens: Iterable[str]) -> List[str]:
    return [w for w in tokens if w and w not in STOP_WORDS]


def lemmatise(tokens: Iterable[str]) -> List[str]:
    return [LEMMATIZER.lemmatize(w) for w in tokens]


def join_tokens(tokens: Iterable[str]) -> str:
    return " ".join(tokens)


def clean_comment(text: str) -> Tuple[str, List[str]]:
    lowered = lowercase(text)
    no_punct = remove_punctuation(lowered)
    tokens = tokenize(no_punct)
    tokens = remove_stopwords(tokens)
    tokens = lemmatise(tokens)
    cleaned = join_tokens(tokens)
    return cleaned, tokens


## 4. Run the Preprocessing Pipeline

This step materialises intermediate artefacts that mirror the original script. Besides the final cleaned text we also persist TF–IDF matrices and helper CSV files so that downstream analysis can reuse them if needed.


In [ ]:
processed_records = []
lemmatised_tokens = []

for comment in df_raw["comment_text"].astype(str):
    cleaned, tokens = clean_comment(comment)
    processed_records.append(cleaned)
    lemmatised_tokens.append(tokens)

preprocessed_df = pd.DataFrame({
    "comment": processed_records,
    "tokens": lemmatised_tokens,
    "toxic": df_raw["toxic"].values,
})

preprocessed_df.to_csv(ARTIFACTS_DIR / "final_preprocessed.csv", index=False)
preprocessed_df.head()


We also inspect token counts to ensure the cleaning behaves as expected.


In [ ]:
word_counts = preprocessed_df["comment"].apply(lambda text: len(text.split()))
plt.figure(figsize=(8, 4))
plt.hist(word_counts, bins=50, color="#4C72B0")
plt.title("Word Count Distribution After Cleaning")
plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "wordcount_hist.png", bbox_inches="tight")
plt.show()


## 5. TF–IDF Feature Extraction

The notebook limits the vocabulary to 5,000 terms (minimum document frequency of 2) to remain faithful to the Python package version.


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, min_df=2)
X_tfidf = vectorizer.fit_transform(preprocessed_df["comment"].values)
y = preprocessed_df["toxic"].values

sparse.save_npz(ARTIFACTS_DIR / "X_tfidf.npz", X_tfidf)
pd.DataFrame({"toxic": y}).to_csv(ARTIFACTS_DIR / "y.csv", index=False)

print("TF-IDF matrix shape:", X_tfidf.shape)


## 6. Train/Test Split

We stratify the split to preserve the 50/50 class balance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape


## 7. Evaluation Helpers

Utility functions for metric computation, ROC handling, confusion-matrix plotting, and the reusable training loop are defined below.


In [ ]:
def get_scores_for_roc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def compute_metrics(y_true, y_pred, y_score=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
    }
    if y_score is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        except ValueError:
            metrics["roc_auc"] = np.nan
    return metrics


def summarise_metrics(model_name, phase, metrics):
    row = {"model": model_name, "phase": phase}
    row.update(metrics)
    return row


def display_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(values_format='d', cmap='Blues')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def display_roc_curve(y_true, y_score, title):
    if y_score is None:
        print("ROC curve skipped: the estimator does not expose calibrated scores.")
        return
    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 8. Train Baseline and Tuned Models

We evaluate six classical algorithms. For each model we capture baseline metrics (default configuration) and tuned metrics using 3-fold cross-validation over the same hyper-parameter grids as the original scripts.


In [ ]:
model_specs = [
    {
        "name": "Logistic Regression",
        "short": "logreg",
        "factory": lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "param_grid": {
            "C": [0.1, 1.0, 5.0],
            "solver": ["liblinear", "lbfgs"],
            "penalty": ["l2"],
        },
    },
    {
        "name": "Multinomial Naive Bayes",
        "short": "nb",
        "factory": lambda: MultinomialNB(),
        "param_grid": {
            "alpha": [0.1, 1.0, 10.0],
            "fit_prior": [True, False],
        },
    },
    {
        "name": "SVM (LinearSVC)",
        "short": "svm",
        "factory": lambda: LinearSVC(random_state=RANDOM_STATE, dual=False),
        "param_grid": {
            "C": [0.5, 1.0, 2.0],
            "loss": ["squared_hinge"],
            "class_weight": [None, "balanced"],
        },
    },
    {
        "name": "Decision Tree",
        "short": "dt",
        "factory": lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "criterion": ["gini", "entropy"],
            "max_depth": [None, 20, 40],
            "min_samples_split": [2, 10, 25],
            "min_samples_leaf": [1, 5, 10],
        },
    },
    {
        "name": "Random Forest",
        "short": "rf",
        "factory": lambda: RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, n_jobs=-1),
        "param_grid": {
            "n_estimators": [200, 400],
            "max_depth": [None, 20, 40],
            "min_samples_split": [2, 10],
            "min_samples_leaf": [1, 5],
        },
    },
    {
        "name": "k-Nearest Neighbours",
        "short": "knn",
        "factory": lambda: KNeighborsClassifier(),
        "param_grid": {
            "n_neighbors": [3, 5, 11],
            "weights": ["uniform", "distance"],
            "metric": ["minkowski", "manhattan"],
        },
    },
]

all_metrics = []
all_best_estimators = {}

for spec in model_specs:
    print(f"
=== {spec['name']} ===")

    baseline_model = spec["factory"]()
    baseline_model.fit(X_train, y_train)
    y_pred = baseline_model.predict(X_test)
    y_score = get_scores_for_roc(baseline_model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    all_metrics.append(summarise_metrics(spec["name"], "baseline", metrics))
    print("Baseline metrics:", metrics)
    display_confusion_matrix(y_test, y_pred, f"{spec['name']} – Baseline")
    display_roc_curve(y_test, y_score, f"{spec['name']} – Baseline ROC")

    param_grid = spec["param_grid"]
    gs = GridSearchCV(
        spec["factory"](),
        param_grid,
        cv=3,
        n_jobs=-1,
        verbose=0,
    )
    gs.fit(X_train, y_train)
    tuned_model = gs.best_estimator_
    all_best_estimators[spec["name"]] = tuned_model
    print("Best hyper-parameters:", gs.best_params_)

    y_pred = tuned_model.predict(X_test)
    y_score = get_scores_for_roc(tuned_model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    all_metrics.append(summarise_metrics(spec["name"], "tuned", metrics))
    print("Tuned metrics:", metrics)
    display_confusion_matrix(y_test, y_pred, f"{spec['name']} – Tuned")
    display_roc_curve(y_test, y_score, f"{spec['name']} – Tuned ROC")

metrics_df = pd.DataFrame(all_metrics)
metrics_df


The consolidated metrics table is saved to `results/` so that it can be inspected outside the notebook as well.


In [ ]:
metrics_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)
metrics_df


## 9. Model Comparison Visualisations

Bar plots help highlight which algorithms benefit most from tuning.


In [ ]:
pivot = metrics_df.pivot_table(index="model", columns="phase", values=["accuracy", "f1", "roc_auc"])
metrics_to_plot = ["accuracy", "f1", "roc_auc"]

for metric in metrics_to_plot:
    plt.figure(figsize=(8, 4))
    metric_data = pivot[metric]
    metric_data.plot(kind="bar", ax=plt.gca())
    plt.title(f"{metric.upper()} – Baseline vs Tuned")
    plt.ylabel(metric.upper())
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{metric}_comparison.png", bbox_inches="tight")
    plt.show()


## 10. Interactive Inference Helper

Select the best tuned estimator (highest F1 score) and expose a convenience function for classifying arbitrary comments.


In [ ]:
# Identify the tuned model with the best F1 score
best_row = metrics_df[metrics_df["phase"] == "tuned"].sort_values("f1", ascending=False).iloc[0]
best_model_name = best_row["model"]
print("Best tuned model:", best_model_name)

best_model = all_best_estimators[best_model_name]


def predict_toxicity(text: str) -> Tuple[int, Optional[float]]:
    cleaned, _ = clean_comment(text)
    X = vectorizer.transform([cleaned])
    if hasattr(best_model, "predict_proba"):
        proba = float(best_model.predict_proba(X)[0, 1])
        return int(proba >= 0.5), proba
    if hasattr(best_model, "decision_function"):
        score = float(best_model.decision_function(X)[0])
        return int(score >= 0.0), score
    pred = int(best_model.predict(X)[0])
    return pred, None

sample_texts = [
    "You are the worst person I have ever met.",
    "Thanks for your help earlier, that was super useful!",
]
for text in sample_texts:
    label, score = predict_toxicity(text)
    print(f"{text}
 -> toxic={label}, score={score}
")


The notebook is intentionally linear: rerun from the top whenever you modify preprocessing or parameter grids to keep every artefact in sync.
